# Bootstrap Coverage: CI and PI
**`fbica` package demo — NonBlock missingness, N=25, T=25**

This notebook replicates the Monte Carlo coverage exercise from the paper at the (25, 25) design point,
using the `fbica` package instead of the raw simulation scripts.

| Type | Target | Bootstrap scheme |
|------|--------|------------------|
| **CI** | Common component $\hat{C}_{i,t,b} = \hat{f}_t' \hat{\lambda}_{i,b}$ | Block-wild pairs bootstrap |
| **PI** | Realised value $x_{i,t,b} = C_{i,t,b} + \nu_{i,t,b}$ | iid pairs bootstrap with residual draw |

> **Tip:** change `phi`, `pi`, or `miss_probs` in *Section 1* to explore the DGP.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from fbica import FBICABootstrap, generate_panel, generate_missing

print('fbica ready ✓')

## 1. Parameters
Edit this cell to explore different DGP configurations.

In [ ]:
# ── Panel dimensions ─────────────────────────────────────────────────
N = 25      # cross-sectional units
T = 25      # time periods
m = 4       # observed variables  (m ≥ r required)
r = 3       # true latent factors

# ── DGP ──────────────────────────────────────────────────────────────
phi = 0.0   # AR(1) persistence in factors and errors  (try 0.3, 0.5)
pi  = 0.0   # spatial dependence coefficient           (try 0.2)

# ── Missingness ───────────────────────────────────────────────────────
miss_probs = np.linspace(0.10, 0.15, m)   # per-variable MCAR rates

# ── Target cells held fixed across all MC replications ───────────────
fixed_points = [(5, 3, 0), (15, 10, 1), (20, 20, 2)]

# ── Bootstrap settings ────────────────────────────────────────────────
B     = 499    # bootstrap replications per simulation
alpha = 0.05   # 1 - alpha = 95% nominal coverage

# ── Monte Carlo ───────────────────────────────────────────────────────
n_sim = 200    # increase to 1 000 for paper-quality results

print(f'N={N}, T={T}, m={m}, r={r}')
print(f'phi={phi}, pi={pi}')
print(f'miss_probs: {miss_probs.round(3)}')
print(f'B={B}, alpha={alpha}, n_sim={n_sim}')

## 2. Fixed DGP elements
Factors **F** and loadings **Λ** are drawn once and held fixed across all MC replications.
Only the idiosyncratic errors are re-drawn each simulation.

In [ ]:
# ── Factors: new-style RNG, seed 99 — matches MC script exactly ──────
rng_fixed = np.random.default_rng(99)
eta = rng_fixed.normal(0, 1, (T, r))
F_fixed = np.zeros((T, r))
F_fixed[0] = eta[0]
for t in range(1, T):
    F_fixed[t] = phi * F_fixed[t - 1] + eta[t]

# ── Loadings: legacy RNG, seed 1 — matches MC script exactly ─────────
np.random.seed(1)
mean_loadings = np.random.normal(0, 1, (m, r))
lam_fixed = np.zeros((N, m, r))
for i in range(N):
    for k in range(m):
        lam_fixed[i, k, :] = mean_loadings[k, :] + np.random.normal(0, 0.5, r)

# True common component (fixed across all sims)
true_C      = np.array([F_fixed[t] @ lam_fixed[i, b] for (t, i, b) in fixed_points])
common_true = np.einsum('tr,ibr->tib', F_fixed, lam_fixed)

# ── Missingness: continue legacy RNG state — matches MC script exactly ─
for _ in range(20):
    fixed_miss = np.zeros((T, N, m), dtype=bool)
    for k in range(m):
        fixed_miss[:, :, k] = np.random.rand(T, N) < miss_probs[k]
    if (not np.any(np.all(fixed_miss, axis=(0, 1))) and
            not np.any(np.all(fixed_miss, axis=(1, 2))) and
            not np.any(np.all(fixed_miss, axis=1))):
        break
fixed_miss = fixed_miss[:, np.random.permutation(N), :]
fixed_miss = fixed_miss[np.random.permutation(T), :, :]

print(f'True C at target cells: {true_C.round(4)}')
print(f'Overall miss rate: {fixed_miss.mean():.2%}')

## 3. Single-sample preview
One realisation — inspect what the CI and PI look like before running the full MC.

In [ ]:
# Uses a new-style RNG so the legacy numpy state (needed by the CI loop) is untouched
rng_preview = np.random.default_rng(0)
nu_demo = rng_preview.normal(0, 1, (T, N, m))
X_demo  = common_true + nu_demo

X_obs_demo = X_demo.copy()
X_obs_demo[fixed_miss] = np.nan
for (t, i, b) in fixed_points:
    X_obs_demo[t, i, b] = np.nan

res_ci = FBICABootstrap(interval_type='CI', B=B, alpha=alpha, seed=0).fit(X_obs_demo, fixed_points)
res_pi = FBICABootstrap(interval_type='PI', B=B, alpha=alpha, seed=0).fit(X_obs_demo, fixed_points)

true_x_demo = np.array([X_demo[t, i, b] for (t, i, b) in fixed_points])

rows = []
for j, (t, i, b) in enumerate(fixed_points):
    rows.append({
        'Cell':      f'(t={t}, i={i}, b={b})',
        'True C':    round(true_C[j], 3),
        'True x':    round(true_x_demo[j], 3),
        'Estimate':  round(res_ci.point_est[j], 3),
        'CI lower':  round(res_ci.lower[j], 3),
        'CI upper':  round(res_ci.upper[j], 3),
        'PI lower':  round(res_pi.lower[j], 3),
        'PI upper':  round(res_pi.upper[j], 3),
    })

pd.DataFrame(rows)

## 4. Monte Carlo — Confidence Intervals
The CI targets the **common component** $C_{i,t,b}$, which is fixed across replications.
Block length is set automatically to $\lceil T^{1/3} \rceil$.

In [ ]:
n_pts = len(fixed_points)
ci_lo  = np.zeros((n_sim, n_pts))
ci_hi  = np.zeros((n_sim, n_pts))
ci_cov = np.zeros((n_sim, n_pts), dtype=int)

for sim in tqdm(range(n_sim), desc='CI bootstrap MC'):
    # Replicate DATA1.DGP1() legacy-numpy draw sequence exactly (phi=0, pi=0)
    np.random.normal(0, 1, (T, r))        # throwaway F draw
    np.random.normal(0, 1, (N, m, r))     # throwaway loading draw
    nu = np.random.normal(0, 1, (T, N, m))
    X  = common_true + nu

    X_obs = X.copy()
    X_obs[fixed_miss] = np.nan
    for (t, i, b) in fixed_points:
        X_obs[t, i, b] = np.nan

    res = FBICABootstrap(interval_type='CI', B=B, alpha=alpha, seed=sim).fit(X_obs, fixed_points)
    ci_lo[sim]  = res.lower
    ci_hi[sim]  = res.upper
    ci_cov[sim] = ((true_C >= res.lower) & (true_C <= res.upper)).astype(int)

In [ ]:
ci_rows = []
for j, (t, i, b) in enumerate(fixed_points):
    ci_rows.append({
        'Cell':       f'(t={t}, i={i}, b={b})',
        'True C':     round(true_C[j], 4),
        'Coverage':   round(ci_cov[:, j].mean(), 4),
        'Left tail':  round((true_C[j] < ci_lo[:, j]).mean(), 4),
        'Right tail': round((true_C[j] > ci_hi[:, j]).mean(), 4),
        'Avg width':  round((ci_hi[:, j] - ci_lo[:, j]).mean(), 4),
    })

print(f'=== CI Coverage | B={B} | n_sim={n_sim} | N={N} | T={T} ===')
pd.DataFrame(ci_rows)

## 5. Monte Carlo — Prediction Intervals
The PI targets the **realised value** $x_{i,t,b}$ (common component + idiosyncratic error),
which is re-drawn every simulation.  PI widths should therefore be larger than CI widths.

In [ ]:
pi_lo  = np.zeros((n_sim, n_pts))
pi_hi  = np.zeros((n_sim, n_pts))
pi_cov = np.zeros((n_sim, n_pts), dtype=int)
true_x = np.zeros((n_sim, n_pts))

rng_mc = np.random.default_rng(42)   # matches PI MC script seed

for sim in tqdm(range(n_sim), desc='PI bootstrap MC'):
    nu = rng_mc.normal(0, 1, (T, N, m))
    X  = common_true + nu
    true_x[sim] = [X[t, i, b] for (t, i, b) in fixed_points]

    X_obs = X.copy()
    X_obs[fixed_miss] = np.nan
    for (t, i, b) in fixed_points:
        X_obs[t, i, b] = np.nan

    res = FBICABootstrap(interval_type='PI', B=B, alpha=alpha, seed=sim).fit(X_obs, fixed_points)
    pi_lo[sim]  = res.lower
    pi_hi[sim]  = res.upper
    pi_cov[sim] = ((true_x[sim] >= res.lower) & (true_x[sim] <= res.upper)).astype(int)

In [ ]:
pi_rows = []
for j, (t, i, b) in enumerate(fixed_points):
    pi_rows.append({
        'Cell':       f'(t={t}, i={i}, b={b})',
        'True C':     round(true_C[j], 4),
        'Coverage':   round(pi_cov[:, j].mean(), 4),
        'Left tail':  round((true_x[:, j] < pi_lo[:, j]).mean(), 4),
        'Right tail': round((true_x[:, j] > pi_hi[:, j]).mean(), 4),
        'Avg width':  round((pi_hi[:, j] - pi_lo[:, j]).mean(), 4),
    })

print(f'=== PI Coverage | B={B} | n_sim={n_sim} | N={N} | T={T} ===')
pd.DataFrame(pi_rows)

## 6. CI vs PI — side-by-side summary

In [ ]:
comp_rows = []
for j, (t, i, b) in enumerate(fixed_points):
    comp_rows.append({
        'Cell':        f'(t={t}, i={i}, b={b})',
        'CI coverage': round(ci_cov[:, j].mean(), 4),
        'CI width':    round((ci_hi[:, j] - ci_lo[:, j]).mean(), 4),
        'PI coverage': round(pi_cov[:, j].mean(), 4),
        'PI width':    round((pi_hi[:, j] - pi_lo[:, j]).mean(), 4),
    })

print(f'N={N}, T={T}, phi={phi}, pi={pi} | B={B} | n_sim={n_sim}')
pd.DataFrame(comp_rows)